In [2]:
# ------ run the SOE loop ------
# ========= SOP-gated drive-cycle replay (US06 @ 10°C) =========
import os, re, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from dataclasses import dataclass
from scipy.signal import butter, lfilter, lfilter_zi
from scipy.interpolate import interp1d
from typing import Optional, List, Dict, Any


@dataclass
class IterDiag:
    iter_idx: int
    Pcand: float
    feasible: bool
    bind: Optional[str]
    V_end: float
    I_end: float
    SOC_end: float
    V_min_step: float
    V_max_step: float
    I_min_step: float
    I_max_step: float


# ------------------- Tunables / params -----------------------------------------
PULSE_TIME = 10            # steps (use 1 Hz cadence)
DT_SEC = 1.0               # seconds per step
NOMINAL_CAPACITY_AH = 60.0 # Ah (cell/pack nominal usable capacity for SOC update)
MAX_ITERS = 50
POWER_TOL_W = 0.1          # stop binary search when bracket < tol (W)

# Electrical limits (pre-margined, i.e., these are the true limits you want enforced)
V_MIN = 2.8
V_MAX = 4.2
I_DISCH_MIN = -300.0       # A (most negative allowed)
I_CHG_MAX   =  180.0       # A (most positive allowed)


# Cell & limits (tweak to your dataset)
NOMINAL_CAPACITY_AH = 60
V_MIN, V_MAX = 2.8, 4.25
I_DISCH_MIN, I_CHG_MAX = -180.0, 120.0
V_EPS, I_EPS = 0.0, 0.0

# Stop policy
POLICY = "stop_on_exceed"   # or "clip_to_sop"
P_FLOOR_W = 60.0           # optional early stop when capability is too low
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
JOB_DIR = "./job_20250922-231821_2_filter_LR_50_Large_2_best_model_filtered_final"
TASK_DIR = "models/FNN_256_128/B4096_LR_SLR_VP300_rep_4"

# Search brackets (adjust to your system power)
P_DISCH_MIN_W = -1500.0
P_DISCH_MAX_W =   0.0
P_CHG_MIN_W   =   0.0
P_CHG_MAX_W   =  1500.0
# ---------------- Utils: model/scaler -----------------
class FNNModel(nn.Module):
    """FNN model matching the training pipeline structure"""
    def __init__(self, input_size, output_size, hidden_layer_sizes_str, activation_str='ReLU', dropout_prob=0.0):
        super(FNNModel, self).__init__()
        
        if isinstance(hidden_layer_sizes_str, str):
            hidden_layer_sizes = [int(size.strip()) for size in hidden_layer_sizes_str.split(',')]
        elif isinstance(hidden_layer_sizes_str, (list, tuple)):
            hidden_layer_sizes = [int(size) for size in hidden_layer_sizes_str]
        else:
            try:
                hidden_layer_sizes = [int(hidden_layer_sizes_str)]
            except:
                hidden_layer_sizes = [256, 128]
        
        layers = []
        current_dim = input_size
        activation_fn = getattr(nn, activation_str, nn.ReLU)

        for hidden_dim in hidden_layer_sizes:
            layers.append(nn.Linear(current_dim, hidden_dim))
            layers.append(activation_fn())
            if dropout_prob > 0:
                layers.append(nn.Dropout(dropout_prob))
            current_dim = hidden_dim
        
        layers.append(nn.Linear(current_dim, output_size))
        self.network = nn.Sequential(*layers)  # <-- MUST be 'network'

    def forward(self, x):
        if x.ndim == 3:
            x = x.view(x.size(0), -1)
        return self.network(x)


def get_hardcoded_scaling_stats():
    return {
        'Temperature':     {'min': -20.310100, 'max': 38.936400, 'scale_factor': 0.01687863},
        'Watts':           {'min': -499.398371, 'max': 276.204152, 'scale_factor': 0.00128932},
        'SOC':             {'min': 0.054946,   'max': 1.000000,   'scale_factor': 1.0},
        'P_filtr_0p002':   {'min': -129.362767,'max': 127.006301, 'scale_factor': 0.00390063},
        'P_filter_0p02':   {'min': -281.917962,'max': 154.761505, 'scale_factor': 0.00229001},
        'Voltage':         {'min': 2.799800,   'max': 4.251300,   'scale_factor': 0.68888896},
    }

def load_model_simple():
    task_info_path = os.path.join(JOB_DIR, TASK_DIR, "task_info.json")
    with open(task_info_path,'r') as f: task_info = json.load(f)
    model_path = os.path.join(JOB_DIR, TASK_DIR, "best_model_export.pt")
    ckpt = torch.load(model_path, map_location=DEVICE)

    if 'hyperparams' in ckpt:
        feature_columns = ckpt['hyperparams'].get('FEATURE_COLUMNS', task_info["hyperparams"]["FEATURE_COLUMNS"])
        hl_raw = ckpt['hyperparams'].get('hidden_layer_sizes', [256,128])
        if isinstance(hl_raw, str): hl_str = hl_raw.strip('[]').replace(' ','')
        elif isinstance(hl_raw, list): hl_str = ','.join(map(str, hl_raw))
        else: hl_str = '256,128'
        act = ckpt['hyperparams'].get('activation','ReLU')
        drop = float(ckpt['hyperparams'].get('dropout_prob',0.2))
        in_size = len(feature_columns)
    else:
        feature_columns = task_info["hyperparams"]["FEATURE_COLUMNS"]
        in_size, hl_str, act, drop = len(feature_columns), '256,128', 'ReLU', 0.2

    model = FNNModel(in_size, 1, hl_str, act, drop).to(DEVICE)
    if 'state_dict' in ckpt: model.load_state_dict(ckpt['state_dict'])
    else:                    model.load_state_dict(ckpt)
    model.eval()
    return model, feature_columns

# --------------- Filter state (for applied P) ---------------
class FilterState:
    def __init__(self, fc_hz=0.002, fs_hz=1.0, order=1, x0=0.0):
        wn = fc_hz / (0.5*fs_hz)
        self.b, self.a = butter(order, wn, btype='low', analog=False)
        self.zi = lfilter_zi(self.b, self.a) * x0

    def step(self, x: float) -> float:
        y, self.zi = lfilter(self.b, self.a, [x], zi=self.zi)
        return float(y[0])

    def clone(self):
        """Return an independent copy with identical internal state."""
        new = object.__new__(FilterState)  # avoid recomputing butter(...)
        new.b = self.b.copy()
        new.a = self.a.copy()
        new.zi = self.zi.copy()
        return new


# --------------- Constraints ------------------------
def check_constraints(v: float, i: float):
    if v < V_MIN - V_EPS or v > V_MAX + V_EPS: return False, 'voltage'
    if i < I_DISCH_MIN - I_EPS or i > I_CHG_MAX + I_EPS: return False, 'current'
    return True, None

@torch.no_grad()
def predict_voltage_fnn(model, scaling_stats, temperature, power, soc, p_filt_0002, p_filt_002):
    """Predict terminal voltage using FNN. Inputs are raw (unscaled)."""
    feats = np.array([temperature, power, soc, p_filt_0002, p_filt_002], dtype=np.float32)
    feature_names = ['Temperature', 'Watts', 'SOC', 'P_filtr_0p002', 'P_filter_0p02']

    x_scaled = np.zeros_like(feats, dtype=np.float32)
    for i, name in enumerate(feature_names):
        x_scaled[i] = (feats[i] - scaling_stats[name]['min']) * scaling_stats[name]['scale_factor']

    x = torch.from_numpy(x_scaled.reshape(1, -1)).to(DEVICE)
    y = model(x)  # normalized voltage
    v_norm = float(y.cpu().numpy()[0, 0])

    v = v_norm / scaling_stats['Voltage']['scale_factor'] + scaling_stats['Voltage']['min']
    return max(v, 0.05)  # small guard

# ------------------- Constraint checks -----------------------------------------
def check_constraints(v: float, i: float):
    """Return (feasible: bool, binding_flag: str|None). Binding flag is 'voltage'/'current'/None."""
    if v < V_MIN - V_EPS or v > V_MAX + V_EPS:
        return False, 'voltage'
    if i < I_DISCH_MIN - I_EPS or i > I_CHG_MAX + I_EPS:
        return False, 'current'
    return True, None

def near_binding(v: float, i: float):
    """Return True if we're sufficiently close to hitting a limit (to accept SOP)."""
    near_v = (V_MIN <= v <= V_MIN + V_EPS) or (V_MAX - V_EPS <= v <= V_MAX)
    near_i = (I_DISCH_MIN + I_EPS <= i <= I_DISCH_MIN + 2*I_EPS) or (I_CHG_MAX - 2*I_EPS <= i <= I_CHG_MAX - I_EPS)
    return near_v or near_i
#Binary search for SOP
def sop_binary_search(
    model,
    scaling_stats,
    temperature: float,
    soc_init: float,
    mode: str,  # 'charge' or 'discharge'
    filt_state_0002,  # FilterState (history up to current step)
    filt_state_002,   # FilterState (history up to current step)
    check_all_steps: bool = True,
    return_diag: bool = False,
):
    """
    Largest constant power over PULSE_TIME*DT_SEC that keeps V/I within limits,
    given *history-aligned* filter states (cloned inside).
    Returns:
        - default: (sop_watts, flag)
        - if return_diag: (sop_watts, flag, iter_logs)
    flag in {'ok','voltage','current','soc','iter'}
    """
    # Bracket by mode
    if mode == 'charge':
        lo, hi = P_CHG_MIN_W, P_CHG_MAX_W
    else:
        lo, hi = P_DISCH_MIN_W, P_DISCH_MAX_W

    # Early SOC sanity
    if (mode == 'discharge' and soc_init <= 0.0) or (mode == 'charge' and soc_init >= 1.0):
        return (0.0, 'soc', []) if return_diag else (0.0, 'soc')

    # Helper: simulate one candidate pulse with cloned history filters
    def simulate_pulse(Pcand: float):
        soc = soc_init
        fs0 = filt_state_0002.clone()
        fs1 = filt_state_002.clone()

        v_hist, i_hist = [], []
        binding = None

        for _ in range(PULSE_TIME):
            p_f0 = fs0.step(Pcand)
            p_f1 = fs1.step(Pcand)

            v = predict_voltage_fnn(model, scaling_stats, temperature, Pcand, soc, p_f0, p_f1)
            i = Pcand / v if v > 0.05 else 0.0

            # SOC update: positive i = charge, negative i = discharge
            soc += (i * DT_SEC) / (3600.0 * NOMINAL_CAPACITY_AH)

            v_hist.append(v); i_hist.append(i)

            feasible, bind = check_constraints(v, i)
            if check_all_steps and (not feasible):
                binding = bind or 'voltage'
                return False, v_hist, i_hist, soc, binding

            if soc <= 0.0 or soc >= 1.0:
                return False, v_hist, i_hist, soc, 'soc'

        if not check_all_steps:
            feasible, bind = check_constraints(v_hist[-1], i_hist[-1])
            if not feasible:
                return False, v_hist, i_hist, soc, bind or 'voltage'

        return True, v_hist, i_hist, soc, None

    # Binary search
    it = 0
    mid = 0.5*(lo + hi)
    last_flag = 'ok'
    iter_logs: List[IterDiag] = []

    while (abs(hi - lo) > POWER_TOL_W) and (it < MAX_ITERS):
        it += 1
        ok, v_hist, i_hist, soc_end, flag = simulate_pulse(mid)
        last_flag = flag or 'ok'

        # Log iteration diagnostics
        iter_logs.append(IterDiag(
            iter_idx=it,
            Pcand=mid,
            feasible=bool(ok),
            bind=flag,
            V_end=float(v_hist[-1]),
            I_end=float(i_hist[-1]),
            SOC_end=float(soc_end),
            V_min_step=float(min(v_hist)),
            V_max_step=float(max(v_hist)),
            I_min_step=float(min(i_hist)),
            I_max_step=float(max(i_hist)),
        ))

        if not ok:
            if mode == 'charge':
                hi = mid
            else:
                lo = mid
        else:
            # Near-binding early exit (tight solution)
            if near_binding(v_hist[-1], i_hist[-1]):
                return (mid, 'ok', iter_logs) if return_diag else (mid, 'ok')
            if mode == 'charge':
                lo = mid
            else:
                hi = mid

        mid = 0.5*(lo + hi)

    # Exit
    if it >= MAX_ITERS:
        return (mid, 'iter', iter_logs) if return_diag else (mid, 'iter')
    return (mid, last_flag, iter_logs) if return_diag else (mid, last_flag)

# --------------- Load & trim US06 -------------------
def load_us06_trim(csv_path, nrows=15890, target_T_C=10.0):
    df = pd.read_csv(csv_path).iloc[:nrows].copy()
    # Robust column detection
    def pick(colnames):
        for c in colnames:
            if c in df.columns: return c
        return None
    colP = pick(['Power','Power_W','Power(W)','P_W','Watts'])
    colSOC = pick(['SOC','soc','SOC_%'])
    colT = pick(['Temperature','Temp','T_C','Aux_Temperature','Aux_Temperature_5(C)'])
    colTime = pick(['Time','Test_Time(s)','t_s','time_s'])
    colV = pick(['Voltage','Voltage(V)','V'])
    colI = pick(['Current','Current(A)','I'])

    if colP is None or colSOC is None:
        raise ValueError("Drive Cycle file must contain power and SOC columns")
    P = df[colP].to_numpy(dtype=float)
    V = df[colV].to_numpy(dtype=float)
    I = df[colI].to_numpy(dtype=float)
    SOC = df[colSOC].to_numpy(dtype=float)
    if 'SOC_%' == colSOC or (SOC.max() > 1.5):  # likely in %
        SOC = SOC / 100.0
    if colT and colT in df.columns:
        T = df[colT].to_numpy(dtype=float)
    else:
        T = np.full_like(P, float(target_T_C), dtype=float)
    if colTime and colTime in df.columns:
        t = df[colTime].to_numpy(dtype=float)
        dt = float(np.median(np.diff(t))) if len(t) > 1 else 1.0
    else:
        dt = 1.0
        t = np.arange(len(P))*dt
    return P, SOC, T, t, dt, V, I


@dataclass
class SimStep:
    k: int
    t_s: float
    mode: str                # 'charge' or 'discharge'
    # demand and application
    P_dem: float
    P_applied: float
    clipped: bool
    # SOP search
    SOP_W: float
    SOP_flag: str
    V_end_from_SOP: float
    # model estimates at applied power
    V_est: float
    I_est: float
    SOC_est: float           # SOC *after* committing step (post-update)
    # experimental (optional)
    V_exp: Optional[float]
    I_exp: Optional[float]
    SOC_exp: Optional[float]
    # margins (model-based)
    Vmin_margin: float
    Vmax_margin: float
    Idis_margin: float
    Ichg_margin: float
    # quick errors (exp - est when exp available)
    dV: Optional[float]
    dI: Optional[float]
    dSOC: Optional[float]
    # bookkeeping
    action: str              # 'apply'|'clip_to_sop'|'zero'|'stop'
    stop_reason: Optional[str]

def simulate_with_sop_gate_bs_logged(
    model, scaling_stats,
    P_d, SOC0, T_C, dt,
    policy='stop_on_exceed', p_floor=100.0,
    # optional experimental traces (same length as P_d) — pass None if unknown
    V_exp: Optional[np.ndarray] = None,
    I_exp: Optional[np.ndarray] = None,
    SOC_exp: Optional[np.ndarray] = None,
    # logging controls
    log_records: bool = True
) -> Dict[str, Any]:
    """
    Online SOP-gated replay with per-sample logging.

    Behavior:
      • SOP is computed at *every* sample (even pre-roll).
      • Before the first non-zero demand (pre-roll), we *log experimental V* (if available)
        and step the filters with 0 W; we do NOT call the model for V at those rows.
      • From the first non-zero demand onward, run the normal SOP-gated path with model V.
      • Discharge exceed -> stop or clip per `policy`; charge exceed -> clip (never stop).
      • Defensive constraint check remains after applying P.
    """
    # --- filter states (history-aligned) ---
    f0 = FilterState(fc_hz=0.002, fs_hz=1.0/dt, order=1, x0=0.0)
    f1 = FilterState(fc_hz=0.02,  fs_hz=1.0/dt, order=1, x0=0.0)

    n   = len(P_d)
    soc = float(SOC0)

    # energy tallies (cell-level)
    E_dis_Wh = 0.0
    E_chg_Wh = 0.0

    # traces
    V_hist = np.zeros(n); I_hist = np.zeros(n); P_app = np.zeros(n)
    sop_arr = np.zeros(n); sop_flag_arr = np.empty(n, dtype=object); Vend_from_sop = np.full(n, np.nan)

    stop_reason, stop_idx = 'end_of_profile', n
    steps: List[SimStep] = []
    iters_long: List[Dict[str, Any]] = []

    # --- pre-roll control ---
    ZERO_EPS = 1e-9
    seen_nonzero = False

    for k in range(n):
        T    = float(T_C[k])
        Pdem = float(P_d[k])
        mode = 'charge' if Pdem > 0.0 else ('discharge' if Pdem < 0.0 else 'charge')

        # -------- SOP (always computed, even during pre-roll) --------
        # Use CLONES of current filter states inside search to preserve history.
        sop, flag, iters = sop_binary_search(
            model, scaling_stats, T, soc, mode,
            filt_state_0002=f0, filt_state_002=f1,
            check_all_steps=True, return_diag=True
        )
                # --- NEW: capture SOP binary-search iterations (best-effort) ---
        if iters:
            for j, it in enumerate(iters):
                row = {
                    "k": k,
                    "iter_idx": j,
                    "mode": mode,
                    "T_C": T,
                    "SOC": soc,
                }
                # best-effort flatten
                if isinstance(it, dict):
                    row.update(it)
                elif hasattr(it, "__dict__"):
                    row.update(it.__dict__)
                else:
                    row["iter_str"] = str(it)
                iters_long.append(row)

        sop_arr[k]      = sop
        sop_flag_arr[k] = flag
        Vend_from_sop[k]= (iters[-1].V_end if iters else np.nan)

        # -------- Pre-roll: demand exactly zero before first non-zero --------
        if (not seen_nonzero) and (abs(Pdem) <= ZERO_EPS):
            # step filters with 0 to keep history aligned
            pf0 = f0.step(0.0); pf1 = f1.step(0.0)

            # use experimental V if provided; otherwise (fallback) model at 0 W
            if V_exp is not None:
                V = float(V_exp[k]); I = 0.0
            else:
                V = predict_voltage_fnn(model, scaling_stats, T, 0.0, soc, pf0, pf1)
                I = 0.0

            # commit traces
            V_hist[k], I_hist[k], P_app[k] = V, I, 0.0

            # log row
            Vexp_k   = None if V_exp  is None else float(V_exp[k])
            Iexp_k   = None if I_exp  is None else float(I_exp[k])
            SOCexp_k = None if SOC_exp is None else float(SOC_exp[k])

            steps.append(SimStep(
                k=k, t_s=k*dt, mode=mode,
                P_dem=Pdem, P_applied=0.0, clipped=False,
                SOP_W=(abs(sop) if mode=='charge' else -abs(sop)),  # sign-normalized SOP for logging
                SOP_flag=flag or 'ok', V_end_from_SOP=float(Vend_from_sop[k]) if Vend_from_sop[k]==Vend_from_sop[k] else np.nan,
                V_est=float(V), I_est=float(I), SOC_est=float(soc),
                V_exp=Vexp_k, I_exp=Iexp_k, SOC_exp=SOCexp_k,
                Vmin_margin=V - V_MIN, Vmax_margin=V_MAX - V,
                Idis_margin=I - I_DISCH_MIN, Ichg_margin=I_CHG_MAX - I,
                dV=(None if Vexp_k is None else Vexp_k - V),
                dI=(None if Iexp_k is None else Iexp_k - I),
                dSOC=(None if SOCexp_k is None else SOCexp_k - soc),
                action='zero', stop_reason=None
            ))
            # proceed to next row without gating
            continue

        # first non-zero demand seen => normal path from now on
        if abs(Pdem) > ZERO_EPS:
            seen_nonzero = True

        # -------- SOP-based gating from here --------
        # normalize SOP sign by mode (discharge negative, charge positive)
        sop_used = abs(sop) if mode == 'charge' else -abs(sop)

        # floor (discharge only)
        floor_hit   = (mode == 'discharge' and abs(sop_used) < p_floor)
        # exceed checks (strict)
        exceeds_dis = (mode == 'discharge' and Pdem < sop_used)   # more negative than SOP ⇒ exceed
        exceeds_chg = (mode == 'charge'    and Pdem > sop_used)   # more positive than SOP ⇒ exceed

        action  = 'apply'
        clipped = False

        if floor_hit:
            stop_reason, stop_idx = 'sop_floor_dis', k
            action = 'stop'
        else:
            if mode == 'discharge':
                if exceeds_dis:
                    if policy == 'stop_on_exceed':
                        stop_reason, stop_idx = 'demand_exceeds_SOP_dis', k
                        action = 'stop'
                    elif policy == 'clip_to_sop':
                        action = 'clip_to_sop'; clipped = True
                    else:
                        raise ValueError(f"Unknown policy {policy}")
            else:  # charge
                if exceeds_chg:
                    action = 'clip_to_sop'; clipped = True

        if action == 'stop':
            # record a stop row (no apply)
            Vexp_k   = None if V_exp  is None else float(V_exp[k])
            Iexp_k   = None if I_exp  is None else float(I_exp[k])
            SOCexp_k = None if SOC_exp is None else float(SOC_exp[k])

            steps.append(SimStep(
                k=k, t_s=k*dt, mode=mode,
                P_dem=Pdem, P_applied=0.0, clipped=False,
                SOP_W=sop_used, SOP_flag=flag or 'ok',
                V_end_from_SOP=float(Vend_from_sop[k]) if Vend_from_sop[k]==Vend_from_sop[k] else np.nan,
                V_est=np.nan, I_est=np.nan, SOC_est=float(soc),
                V_exp=Vexp_k, I_exp=Iexp_k, SOC_exp=SOCexp_k,
                Vmin_margin=np.nan, Vmax_margin=np.nan, Idis_margin=np.nan, Ichg_margin=np.nan,
                dV=None, dI=None, dSOC=(None if SOCexp_k is None else SOCexp_k - soc),
                action='stop', stop_reason=stop_reason
            ))
            break

        # choose applied power (clip or demand)
        P = sop_used if clipped else Pdem

        # advance live filters with applied power only
        pf0 = f0.step(P); pf1 = f1.step(P)

        # model outputs at applied power
        V = predict_voltage_fnn(model, scaling_stats, T, P, soc, pf0, pf1)
        I = P / V if abs(V) > 1e-6 else 0.0

        # defensive constraints
        feasible, bind = check_constraints(V, I)
        if not feasible:
            stop_reason, stop_idx = f'violate_limits:{bind or "voltage"}', k

            Vexp_k   = None if V_exp  is None else float(V_exp[k])
            Iexp_k   = None if I_exp  is None else float(I_exp[k])
            SOCexp_k = None if SOC_exp is None else float(SOC_exp[k])

            steps.append(SimStep(
                k=k, t_s=k*dt, mode=mode,
                P_dem=Pdem, P_applied=P, clipped=clipped,
                SOP_W=sop_used, SOP_flag=flag or 'ok',
                V_end_from_SOP=float(Vend_from_sop[k]) if Vend_from_sop[k]==Vend_from_sop[k] else np.nan,
                V_est=float(V), I_est=float(I), SOC_est=float(soc),
                V_exp=Vexp_k, I_exp=Iexp_k, SOC_exp=SOCexp_k,
                Vmin_margin=V - V_MIN, Vmax_margin=V_MAX - V,
                Idis_margin=I - I_DISCH_MIN, Ichg_margin=I_CHG_MAX - I,
                dV=(None if Vexp_k is None else Vexp_k - V),
                dI=(None if Iexp_k is None else Iexp_k - I),
                dSOC=(None if SOCexp_k is None else SOCexp_k - soc),
                action='violate', stop_reason=stop_reason
            ))
            break

        # commit traces
        V_hist[k], I_hist[k], P_app[k] = V, I, P

        # energy accounting (Wh)
        dE = (P * dt) / 3600.0
        if P < 0:   E_dis_Wh += -dE
        else:       E_chg_Wh +=  dE

        # SOC update
        soc = soc + (I * dt) / (3600.0 * NOMINAL_CAPACITY_AH)
        if soc <= 0.0 or soc >= 1.0:
            stop_reason, stop_idx = 'soc_bound', k + 1

        # experimental values at k (optional)
        Vexp_k   = None if V_exp  is None else float(V_exp[k])
        Iexp_k   = None if I_exp  is None else float(I_exp[k])
        SOCexp_k = None if SOC_exp is None else float(SOC_exp[k])

        steps.append(SimStep(
            k=k, t_s=k*dt, mode=mode,
            P_dem=Pdem, P_applied=P, clipped=clipped,
            SOP_W=sop_used, SOP_flag=flag or 'ok',
            V_end_from_SOP=float(Vend_from_sop[k]) if Vend_from_sop[k]==Vend_from_sop[k] else np.nan,
            V_est=float(V), I_est=float(I), SOC_est=float(soc),
            V_exp=Vexp_k, I_exp=Iexp_k, SOC_exp=SOCexp_k,
            Vmin_margin=V - V_MIN, Vmax_margin=V_MAX - V,
            Idis_margin=I - I_DISCH_MIN, Ichg_margin=I_CHG_MAX - I,
            dV=(None if Vexp_k is None else Vexp_k - V),
            dI=(None if Iexp_k is None else Iexp_k - I),
            dSOC=(None if SOCexp_k is None else SOCexp_k - soc),
            action=('clip_to_sop' if clipped else ('apply' if abs(Pdem) > ZERO_EPS else 'zero')),
            stop_reason=(None if stop_idx==n else stop_reason)
        ))

        if stop_reason != 'end_of_profile':
            break

    # -------- Trim arrays to executed horizon --------
    V_hist        = V_hist[:stop_idx]
    I_hist        = I_hist[:stop_idx]
    P_app         = P_app[:stop_idx]
    sop_arr       = sop_arr[:stop_idx]
    sop_flag_arr  = sop_flag_arr[:stop_idx]
    Vend_from_sop = Vend_from_sop[:stop_idx]

    # -------- Energy & range (cell-level) --------
    E_UBE_Wh = E_dis_Wh - E_chg_Wh
    range_km = (E_dis_Wh / 1000.0) * VEH_KM_PER_KWH

    # -------- Assemble outputs --------
    out = {
        'E_dis_Wh'      : E_dis_Wh,
        'E_chg_Wh'      : E_chg_Wh,
        'E_UBE_Wh'      : E_UBE_Wh,
        'range_km'      : range_km,
        'stop_reason'   : stop_reason,
        'stop_index'    : stop_idx,
        'V'             : V_hist,
        'I'             : I_hist,
        'P_applied'     : P_app,
        'SOP'           : sop_arr,
        'SOP_flag'      : sop_flag_arr,
        'V_end_from_SOP': Vend_from_sop,
        'soc_final'     : soc,
        'iters_long'    : iters_long,
    }

    if log_records:
        out['records'] = steps
        try:
            import pandas as pd
            df = pd.DataFrame([s.__dict__ for s in steps])
            for col in ('dV','dI','dSOC'):
                if col in df.columns and df[col].notna().any():
                    df[f'abs_{col}'] = df[col].abs()
            out['records_df'] = df
        except Exception:
            out['records_df'] = None

    return out


# ----------------------- CONFIG -----------------------

#DC_PATH = "/mnt/data1/dehuryb/SOE/SOP/UDDS_25_degC.csv"
DC_PATH = "./UDDS_25_degC.csv"# <- adjust if needed
#TRIM_ROWS = 15230 # US06_0
TRIM_ROWS = 46866 #UDDS_25
DC_TEMP = 25
#LEAN_LUT = "/mnt/data1/dehuryb/SOE/SOP/output_tables/FNN_SOP_LUT_all_temp_lean.csv"


# ================== RUN (Online SOP per step) ==================
# ================== BATCH RUN (Online SOP per step) ==================
VEH_KM_PER_KWH = 6.5  # km/kWh assumption for range

# --- Output dirs ---
STEP_DIR = "step_dfs"
ITER_DIR = "iter_logs"
SUMM_DIR = "summaries"
os.makedirs(STEP_DIR, exist_ok=True)
os.makedirs(ITER_DIR, exist_ok=True)
os.makedirs(SUMM_DIR, exist_ok=True)

# --- Choose which files to run (edit this dict as you like) ---
# trim_rows can be None to run full file
temps  = [-20, -10, 0, 10, 25, 40]
cycles = ['HWFET', 'LA92', 'UDDS', 'US06']

rngStart = [
    [5000, 5300,    0, 6200],   # -20
    [4500, 4400,    0, 5600],   # -10
    [3368, 2400,    0, 3500],   # 0
    [1750, 2250,    0, 2170],   # 10
    [ 900,  900,    0,  850],   # 25
    [5800, 4500,    0, 6500],   # 40
]

rngEnd = [
    [23600, 32000, 37800, 18400],
    [22700, 32600, 42500, 18000],
    [22450, 33400, 47200, 17600],
    [22000, 33500, 48900, 16200],
    [19800, 31900, 48300, 14800],
    [27500, 38900, 48180, 22000],
]
RUNS = {}

for ti, T in enumerate(temps):
    for ci, cyc in enumerate(cycles):
        start = rngStart[ti][ci]
        end   = rngEnd[ti][ci]

        # skip invalid entries if you want (optional)
        if end <= start:
            continue

        # filename convention
        if T < 0:
            fname = f"{cyc}_n{abs(T)}_degC.csv"
        else:
            fname = f"{cyc}_{T}_degC.csv"

        RUNS[fname] = {
            "temp": T,
            "row_start": int(start),
            "row_end": int(end),
        }


# --- Model & scaling (load once) ---
model, feature_cols = load_model_simple()
scaling_stats = get_hardcoded_scaling_stats()

# --- Policy & floor (global) ---
POLICY = 'stop_on_exceed'   # 'clip_to_sop' or 'stop_on_exceed'
P_FLOOR_W = 80.0

# --- Pack topology ---
NS, NP = 96, 2
SCALE = NS * NP

summary_rows = []

for DC_PATH, cfg in RUNS.items():
    if not os.path.exists(DC_PATH):
        print(f"[SKIP] missing: {DC_PATH}")
        continue

    DC_TEMP   = float(cfg["temp"])
    row_start = int(cfg["row_start"])
    row_end   = int(cfg["row_end"])
    
    print(f"\n=== Running {DC_PATH} @ {DC_TEMP:+.0f}°C | rows [{row_start}:{row_end}] ===")
    
    # load full file (or you *can* load nrows=row_end to save time)
    P_d, SOC_exp, T_C, t, dt, V, I = load_us06_trim(DC_PATH, nrows=row_end, target_T_C=DC_TEMP)  
    # slice to window
    P_d     = P_d[row_start:row_end]
    SOC_exp = SOC_exp[row_start:row_end] if SOC_exp is not None else None
    T_C     = T_C[row_start:row_end]
    t       = t[row_start:row_end]
    V       = V[row_start:row_end]
    I       = I[row_start:row_end]

    # experimental signals available in file
    V_exp, I_exp = V, I

    SOC0 = float(SOC_exp[0]) if SOC_exp is not None and len(SOC_exp) > 0 else 0.80

    res = simulate_with_sop_gate_bs_logged(
        model, scaling_stats,
        P_d=P_d, SOC0=SOC0, T_C=T_C, dt=dt,
        policy=POLICY, p_floor=P_FLOOR_W,
        V_exp=V_exp, I_exp=I_exp, SOC_exp=SOC_exp,
        log_records=True
    )

    steps_df = res.get('records_df', None)

    # ---------- Save steps_df ----------
    tag = os.path.splitext(os.path.basename(DC_PATH))[0]  # e.g., UDDS_25_degC
    steps_path = os.path.join(STEP_DIR, f"steps_{tag}.csv")
    if steps_df is not None:
        steps_df.to_csv(steps_path, index=False)
    else:
        steps_path = ""

    # ---------- Save iter logs (requires the tiny patch above) ----------
    iters_path = ""
    if "iters_long" in res and res["iters_long"] is not None and len(res["iters_long"]) > 0:
        iters_df = pd.DataFrame(res["iters_long"])
        iters_path = os.path.join(ITER_DIR, f"iters_{tag}.csv")
        iters_df.to_csv(iters_path, index=False)

    # ---------- Compute pack metrics ----------
    end_time_s = res['stop_index'] * dt

    cell_E_dis_Wh = res['E_dis_Wh']
    cell_E_chg_Wh = res['E_chg_Wh']
    cell_E_UBE_Wh = res['E_UBE_Wh']

    pack_E_dis_Wh = cell_E_dis_Wh * SCALE
    pack_E_chg_Wh = cell_E_chg_Wh * SCALE
    pack_E_UBE_Wh = cell_E_UBE_Wh * SCALE

    pack_range_from_dis_km = (pack_E_dis_Wh / 1000.0) * VEH_KM_PER_KWH
    pack_range_from_ube_km = (pack_E_UBE_Wh / 1000.0) * VEH_KM_PER_KWH

    # console print (similar to your single-run)
    print(f"Stop reason             : {res['stop_reason']}")
    print(f"End time                : {end_time_s:.1f} s (index {res['stop_index']})")
    print(f"SOC final (cell)        : {res['soc_final']:.4f}")
    print(f"Pack range (discharge)  : {pack_range_from_dis_km:.2f} km")

    # ---------- Save per-run summary row ----------
    summary_rows.append({
        "file": DC_PATH,
        "temp_C": DC_TEMP,
        "trim_rows_used": len(P_d),
        "row_start": row_start,
        "row_end": row_end,
        "dt_s": dt,
        "policy": POLICY,
        "p_floor_W": P_FLOOR_W,
        "stop_reason": res["stop_reason"],
        "stop_index": res["stop_index"],
        "end_time_s": end_time_s,
        "soc0": SOC0,
        "soc_final": res["soc_final"],
        "cell_E_dis_Wh": cell_E_dis_Wh,
        "cell_E_chg_Wh": cell_E_chg_Wh,
        "cell_E_UBE_Wh": cell_E_UBE_Wh,
        "pack_E_dis_kWh": pack_E_dis_Wh / 1000.0,
        "pack_E_chg_kWh": pack_E_chg_Wh / 1000.0,
        "pack_E_UBE_kWh": pack_E_UBE_Wh / 1000.0,
        "pack_range_from_dis_km": pack_range_from_dis_km,
        "pack_range_from_ube_km": pack_range_from_ube_km,
        "steps_csv": steps_path,
        "iters_csv": iters_path,
    })

# write master summary
summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(SUMM_DIR, "batch_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print(f"\nSaved: {summary_csv}")
print(f"Steps: {STEP_DIR}/")
print(f"Iters: {ITER_DIR}/ (only if you applied the tiny patch)")


=== Running HWFET_n20_degC.csv @ -20°C | rows [5000:23600] ===
Stop reason             : demand_exceeds_SOP_dis
End time                : 14379.0 s (index 14379)
SOC final (cell)        : 0.2731
Pack range (discharge)  : 219.95 km

=== Running LA92_n20_degC.csv @ -20°C | rows [5300:32000] ===
Stop reason             : demand_exceeds_SOP_dis
End time                : 18951.0 s (index 18951)
SOC final (cell)        : 0.3679
Pack range (discharge)  : 258.97 km

=== Running UDDS_n20_degC.csv @ -20°C | rows [0:37800] ===
Stop reason             : demand_exceeds_SOP_dis
End time                : 30336.0 s (index 30336)
SOC final (cell)        : 0.3166
Pack range (discharge)  : 296.96 km

=== Running US06_n20_degC.csv @ -20°C | rows [6200:18400] ===
Stop reason             : demand_exceeds_SOP_dis
End time                : 7132.0 s (index 7132)
SOC final (cell)        : 0.4582
Pack range (discharge)  : 189.88 km

=== Running HWFET_n10_degC.csv @ -10°C | rows [4500:22700] ===
Stop reason     